# Portfolio and Risk Management
## Homework 1: HMC and Inflation-Indexed Bonds
### Group A


In [12]:
import pandas as pd 
path = "data/multi_asset_etf_data.xlsx"
excess_returns = pd.read_excel(path, sheet_name="excess returns", index_col=0)
excess_returns = excess_returns.drop(columns = "QAI")

# 2 Mean-Variance Optimization
## 2.1 Summary Statistics

In [13]:
import numpy as np
annualized_mean_returns = excess_returns.mean() * 12
annualized_return_volatility = excess_returns.std() * np.sqrt(12)
sharpe_ratio = annualized_mean_returns / annualized_return_volatility
summary_stats = pd.DataFrame({"annualized_mean": annualized_mean_returns, "annualized volatility": annualized_return_volatility, "sharpe ratio": sharpe_ratio})
print(summary_stats)

     annualized_mean  annualized volatility  sharpe ratio
BWX        -0.017163               0.081891     -0.209582
DBC         0.013000               0.172276      0.075461
EEM         0.044426               0.179085      0.248074
EFA         0.063754               0.148590      0.429057
HYG         0.034500               0.073326      0.470501
IEF         0.008596               0.062120      0.138377
IYR         0.070095               0.165531      0.423456
PSP         0.076428               0.211323      0.361666
SPY         0.126955               0.141398      0.897856
TIP         0.012776               0.049748      0.256817


### Which assets have the best and worse Sharpe Ratios?

The best Sharpe Ratios is the highest one since that represents more return for a certain level of risk, therefore the asset with the best Sharpe Ratio is SPY with a ratio of 0.898. The worst ratio is the lowest ratio which is BWX with the Sharpe Ratio of -0.20958. As you can see in the table above, BWX has a negative annualized mean which is what causes the ratio to be negative. 

## 2.2 Descriptive Analysis

In [14]:
correlation_matrix = excess_returns.corr()
print(correlation_matrix)

          BWX       DBC       EEM       EFA       HYG       IEF       IYR  \
BWX  1.000000  0.160774  0.611691  0.612692  0.600427  0.580844  0.555010   
DBC  0.160774  1.000000  0.438794  0.440497  0.418825 -0.308778  0.252598   
EEM  0.611691  0.438794  1.000000  0.811959  0.669611  0.051462  0.574328   
EFA  0.612692  0.440497  0.811959  1.000000  0.776442  0.062915  0.700449   
HYG  0.600427  0.418825  0.669611  0.776442  1.000000  0.192726  0.731282   
IEF  0.580844 -0.308778  0.051462  0.062915  0.192726  1.000000  0.321839   
IYR  0.555010  0.252598  0.574328  0.700449  0.731282  0.321839  1.000000   
PSP  0.523903  0.420190  0.697394  0.861604  0.798316  0.012493  0.726034   
SPY  0.443512  0.383582  0.693457  0.832782  0.783147  0.011276  0.743470   
TIP  0.675024  0.084279  0.383897  0.402497  0.540414  0.756082  0.599880   

          PSP       SPY       TIP  
BWX  0.523903  0.443512  0.675024  
DBC  0.420190  0.383582  0.084279  
EEM  0.697394  0.693457  0.383897  
EFA  0.8

In [15]:
asset_classes = correlation_matrix.columns
result = []


for i in range(len(asset_classes)):
    for j in range(i + 1, len(asset_classes)):        
        
        asset1 = asset_classes[i]
        asset2 = asset_classes[j]
        selected_corr = correlation_matrix.loc[asset1, asset2]

        result.append((asset1, asset2, selected_corr))

asset_pairs = pd.DataFrame(result, columns = ["asset1", "asset2", "correlation coefficient"]).sort_values("correlation coefficient", key=abs)
# print(asset_pairs)
highest = asset_pairs.head(1)
lowest = asset_pairs.tail(1)
print(highest), print(lowest)

   asset1 asset2  correlation coefficient
37    IEF    SPY                 0.011276
   asset1 asset2  correlation coefficient
42    PSP    SPY                 0.872151


(None, None)

Highest correlation is between IEF and SPY with a correlation coefficient of 0.011276
Lowest Correlation is between PSP and SPY with a correlation coefficient of 0.8722


In [16]:
print(summary_stats.loc[["TIP", "IEF", "BWX"]])
print(correlation_matrix.loc["TIP", ["IEF", "BWX"]])

     annualized_mean  annualized volatility  sharpe ratio
TIP         0.012776               0.049748      0.256817
IEF         0.008596               0.062120      0.138377
BWX        -0.017163               0.081891     -0.209582
IEF    0.756082
BWX    0.675024
Name: TIP, dtype: float64


In our sample, TIPS outperforms Domestic Bonds and Foreign Bonds

## 2.3 The MV Frontier

Note that $$w^{\mathrm{tan}} = \delta_{\mathrm{tan}} \Sigma^{-1} \tilde{\mu}$$
so we can use $x = \Sigma^{-1} \tilde{\mu}$ and solve $\Sigma x = \tilde{\mu}$

In [17]:
cov_matrix = excess_returns.cov()

x = np.linalg.solve(cov_matrix, excess_returns.mean())
wtan = x/(x.sum())
tickers = excess_returns.columns.to_numpy()

tangency = list(zip(tickers, wtan, sharpe_ratio))
tangency.sort(key = lambda tangency: tangency[1], reverse = True)

print(f"{'Ticker':<10}{'Weight':>12}{'Sharpe':>12}")
print("-" * 34)
for ticker, weight, sharpe in tangency: 
    print(f"{ticker:<10}{weight:>12.2%}{sharpe:>12.2%}")


Ticker          Weight      Sharpe
----------------------------------
IEF            131.88%      13.84%
SPY            126.32%      89.79%
HYG             20.38%      47.05%
EFA             18.48%      42.91%
DBC              3.63%       7.55%
EEM             -0.10%      24.81%
TIP            -19.20%      25.68%
IYR            -22.37%      42.35%
PSP            -41.06%      36.17%
BWX           -117.97%     -20.96%


The ranking of tangency weights does not align with the ranking of Sharpe ratios. 

In [18]:
tan_mean = wtan @ annualized_mean_returns
tan_vol = np.sqrt(wtan @ (cov_matrix *12) @ wtan)
tan_sharpe = tan_mean / tan_vol

print("Tangency Portfolio") 
print("-" * 23)
print(f"{'Mean':<10} {tan_mean:>12.2%}")
print(f"{'Volatility':<10} {tan_vol:>12.2%}")
print(f"{'Sharpe':<10} {tan_sharpe:>12.2%}")

Tangency Portfolio
-----------------------
Mean             16.17%
Volatility       10.96%
Sharpe          147.58%


## 2.4 TIPS

In [19]:
def tangency_portfolio(mean_return, cov_matrix):
    x = pd.Series(
        np.linalg.solve(cov_matrix.values, mean_return.values),
        index=mean_return.index
    )
    
    weights = x / x.sum()
    
    ann_mean = 12 * (weights @ mean_return)
    ann_vol = np.sqrt(12 * (weights @ cov_matrix @ weights))
    sharpe = ann_mean / ann_vol
    
    return weights, ann_mean, ann_vol, sharpe

In [20]:
mean_return = excess_returns.mean()
cov_matrix = excess_returns.cov()

w_base, mean_base, vol_base, sharpe_base = tangency_portfolio(
    mean_return, cov_matrix
)

#### Drop TIPS:

In [21]:
#Drop TIPS
returns_no_tip = excess_returns.drop(columns="TIP")

mean_no_tip = returns_no_tip.mean()
cov_no_tip = returns_no_tip.cov()

w_no_tip, mean_no_tip_port, vol_no_tip, sharpe_no_tip = tangency_portfolio(
    mean_no_tip, cov_no_tip
)

for ticker, weight in w_no_tip.items():
    print(f"{ticker:<10}{weight:>12.2%}")

print()
print(f"{'mean':<12}{mean_no_tip_port:>12.2%}")
print(f"{'volatility':<12}{vol_no_tip:>12.2%}")
print(f"{'Sharpe':<12}{sharpe_no_tip:>12.3f}")

BWX           -113.75%
DBC              2.58%
EEM             -0.62%
EFA             18.81%
HYG             17.91%
IEF            116.16%
IYR            -22.34%
PSP            -40.04%
SPY            121.28%

mean              15.54%
volatility        10.54%
Sharpe             1.475


#### Increase expected TIPS return by 0.0012:

In [22]:
# increase expected TIPS return by 0.0012
mean_tip_adjusted = mean_return.copy()
mean_tip_adjusted["TIP"] += 0.0012

w_tip_adjusted, mean_tip_adj, vol_tip_adj, sharpe_tip_adj = tangency_portfolio(
    mean_tip_adjusted, cov_matrix
)


for ticker, weight in w_tip_adjusted.items():
    print(f"{ticker:<10}{weight:>12.2%}")

print()
print(f"{'mean':<12}{mean_tip_adj:>12.2%}")
print(f"{'volatility':<12}{vol_tip_adj:>12.2%}")
print(f"{'Sharpe':<12}{sharpe_tip_adj:>12.3f}")

# Ultimately, it seems like TIPS don't have a meaningful impact on the investment opportunity set; 
# dropping TIPS only reduces the tangency protfolio's Sharpe ration from 1.476 to 1.475. 
# Increasing TIPS' expected monthly excess returns by 0.0012 caused their weight to rise by about 126% 
# and increased the tangency Sharpe ratio to 1.575. THerefore, it seems like TIPS can materially expand 
# investment opportunity set iff expected return is somehwat higher than historical samples suggest. 
# I think that Harvard, then, should consider TIPS as a seperate asset class.

BWX            -86.02%
DBC             -4.33%
EEM             -4.02%
EFA             20.94%
HYG              1.72%
IEF             12.89%
IYR            -22.16%
PSP            -33.38%
SPY             88.19%
TIP            126.17%

mean              13.27%
volatility         8.42%
Sharpe             1.575


Ultimately, it seems like TIPS don't have a meaningful impact on the investment opportunity set; 
- Dropping TIPS only reduces the tangency protfolio's Sharpe ration from 1.476 to 1.475. 
- Increasing TIPS' expected monthly excess returns by 0.0012 caused their      weight to rise by about 126% and increased the tangency Sharpe ratio to 1.575.

Therefore, it seems like TIPS can materially expand investment opportunity set iff expected return is somehwat higher than historical samples suggest.
 
I think that Harvard, then, should consider TIPS as a seperate asset class.

# 3 Allocations

Next, the EW, RP, and MV portfolios are calculated, and their weight vectors are rescaled to achieve an excess return (per month) of $\tilde{\mu}^{\mathrm{port}} = 0.01$.

In [23]:
target_mean = 0.01
mean_returns = excess_returns.mean()

# Equally-weighted (EW)
n = excess_returns.shape[1]
w_ew_base = np.ones(n) / n
w_ew = w_ew_base * (target_mean / (w_ew_base @ mean_returns))

# Risk-parity (RP)
w_rp_base = 1 / excess_returns.var().to_numpy()
w_rp = w_rp_base * (target_mean / (w_rp_base @ mean_returns))

# Mean-Variance (MV)
w_mv_base = wtan
w_mv = w_mv_base * (target_mean / (w_mv_base @ mean_returns))

weights = pd.DataFrame({"EW": w_ew, "RP": w_rp, "MV": w_mv,}, index=excess_returns.columns,)

print("Portfolio weights after rescaling to a 1% monthly mean excess return:")
print(weights)

Portfolio weights after rescaling to a 1% monthly mean excess return:
           EW        RP        MV
BWX  0.276901  0.673280 -0.875565
DBC  0.276901  0.152131  0.026945
EEM  0.276901  0.140783 -0.000724
EFA  0.276901  0.204497  0.137192
HYG  0.276901  0.839748  0.151256
IEF  0.276901  1.170044  0.978827
IYR  0.276901  0.164783 -0.166009
PSP  0.276901  0.101106 -0.304741
SPY  0.276901  0.225831  0.937525
TIP  0.276901  1.824379 -0.142495


Performance of each of these portfolios over the sample and their summary statistics.

In [24]:
portfolio_returns = excess_returns @ weights

performance = pd.DataFrame(
    {"Mean": portfolio_returns.mean() * 12,
     "Volatility": portfolio_returns.std() * np.sqrt(12),
     "Sharpe": portfolio_returns.mean() / portfolio_returns.std() * np.sqrt(12),})

print("Annualized performance:")
print(performance)

Annualized performance:
    Mean  Volatility    Sharpe
EW  0.12    0.275296  0.435894
RP  0.12    0.326142  0.367938
MV  0.12    0.081314  1.475763


### How does performance compare across allocation methods?

The Minimum-Variance portfolio has the highest Sharpe ratio, as it achieves the lowest volatility among the three portfolios while maintaining the same target return. Since all portfolios are rescaled to the same target mean excess return, differences in Sharpe ratios are driven by their differences in volatility.

The Equal-Weighted portfolio has lower volatility than the Risk-Parity portfolio (27.5% versus 32.6%). On the other hand, compared to MV, EW and RP assign positive weights to all assets, whereas the unconstrained MV portfolio includes both positive and negative weights. This reflects the different allocation mechanisms: EW and RP impose positive weights by construction, while the unconstrained MV optimization allows short positions.